In [ ]:
import os, re, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from sklearn.model_selection import train_test_split
from evaluate import load
from lime.lime_text import LimeTextExplainer


In [2]:
# Load GloVe vectors again
GLOVE_PATH = r"C:\Users\bhoom\Downloads\glove.6B.300d.txt"
glove_dim = 300
glove_dict = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]
        vec = np.asarray(parts[1:], dtype=np.float32)
        glove_dict[word] = vec

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab = tokenizer.get_vocab()
vocab_size = len(vocab)

emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, glove_dim)).astype(np.float32)
for tok, idx in vocab.items():
    key = tok if tok in glove_dict else tok.lower()
    if key in glove_dict:
        emb_matrix[idx] = glove_dict[key]
emb_tensor = torch.from_numpy(emb_matrix)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

C:\Users\bhoom\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhoom\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [3]:
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row.question,
            row.answer,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        input_ids = enc.input_ids.squeeze()
        attention_mask = enc.attention_mask.squeeze()
        labels = input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100
        return input_ids, attention_mask, labels

def collate_fn(batch):
    ids, masks, labs = zip(*batch)
    return torch.stack(ids), torch.stack(masks), torch.stack(labs)


In [4]:
df_passages = pd.read_parquet(
    r"hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet"
)
df_test = pd.read_parquet(
    r"hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet"
)

# 2. Cleaning function
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', str(text))
    return text.strip().lower()

# 3. Apply cleaning
df_passages['passage'] = df_passages['passage'].apply(clean_text)
df_test['question'] = df_test['question'].apply(clean_text)
df_test['answer']   = df_test['answer'].apply(clean_text)

# 4. Prepare (question, answer) pairs
qa_df = pd.DataFrame({
    'question': df_test['question'],
    'answer':   df_test['answer']
}).query("answer.str.len() > 0", engine='python')

# 5. Save to CSV
output_path = r"C:\Users\bhoom\Downloads\qa_pairs.csv"
qa_df.to_csv(output_path, index=False)
print(f"✅ Saved {len(qa_df)} QA pairs to {output_path}")

✅ Saved 4719 QA pairs to C:\Users\bhoom\Downloads\qa_pairs.csv


In [5]:
path = r"C:\Users\bhoom\Downloads\qa_pairs.csv"
qa_df = pd.read_csv(path)

# 2. Drop duplicate rows
before = len(qa_df)
qa_df = qa_df.drop_duplicates(subset=['question', 'answer'])
after = len(qa_df)
print(f"Dropped {before - after} duplicate pairs; {after} remain.")

# 3. Split into train/validation (80% train, 20% val)
qa_train, qa_val = train_test_split(
    qa_df, test_size=0.2, random_state=42, shuffle=True
)

# 4. Save splits
train_path = r"C:\Users\bhoom\Downloads\qa_train.csv"
val_path   = r"C:\Users\bhoom\Downloads\qa_val.csv"
qa_train.to_csv(train_path, index=False)
qa_val.to_csv(val_path,   index=False)

print(f"Train pairs: {len(qa_train)} → {train_path}")
print(f" Val pairs: {len(qa_val)} → {val_path}")

Dropped 0 duplicate pairs; 4719 remain.
Train pairs: 3775 → C:\Users\bhoom\Downloads\qa_train.csv
 Val pairs: 944 → C:\Users\bhoom\Downloads\qa_val.csv


In [6]:
qa_train = pd.read_csv(r"C:\Users\bhoom\Downloads\qa_train.csv")
qa_val   = pd.read_csv(r"C:\Users\bhoom\Downloads\qa_val.csv")

train_ds = QADataset(qa_train, tokenizer, max_len=128)
val_ds   = QADataset(qa_val, tokenizer, max_len=128)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=16, collate_fn=collate_fn)


In [15]:
import torch
import torch.nn as nn

class CausalTransformer(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, n_heads, n_layers, max_len, dropout):
        super(CausalTransformer, self).__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float), freeze=False
        )
        self.position_embedding = nn.Embedding(max_len, embedding_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        seq_len = input_ids.size(1)
        positions = torch.arange(0, seq_len, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.position_embedding(positions)
        x = self.transformer(x)
        return self.output_layer(x)


In [23]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
import re

# --------- 1. Load Tokenizer ---------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"pad_token": "[PAD]"})

# --------- 2. Load GloVe ---------
#GLOVE_DIR  = r"C:\Users\ADVAIT\Desktop\NLP A3\glove.6B"
#GLOVE_FILE = "glove.6B.300d.txt"  # adjust dim if needed
GLOVE_PATH = r"C:\Users\bhoom\Downloads\glove.6B.300d.txt"
embedding_dim = 300  # match glove file

print("Loading GloVe embeddings...")
glove_dict = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]
        vec = np.asarray(parts[1:], dtype=np.float32)
        glove_dict[word] = vec

# --------- 3. Build Embedding Matrix ---------
vocab = tokenizer.get_vocab()
vocab_size = len(vocab)
print(f"Tokenizer vocab size: {vocab_size}")

embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embedding_dim)).astype(np.float32)
for tok, idx in vocab.items():
    key = tok if tok in glove_dict else tok.lower()
    if key in glove_dict:
        embedding_matrix[idx] = glove_dict[key]
print("Embedding matrix ready")

# --------- 4. Dataset (dummy example) ---------
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row['question'],
            row['answer'],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        input_ids = enc.input_ids.squeeze(0)
        attention_mask = enc.attention_mask.squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return input_ids, attention_mask, labels

# --------- 5. Model Definition ---------
class CausalTransformer(nn.Module):
    def __init__(self, embedding_matrix, num_heads, num_layers, max_len, dropout):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embed_dim = embed_dim
        self.token_emb = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out = nn.Linear(embed_dim, vocab_size)
        self.max_len = max_len

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.size()
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        # Transformer expects no causal mask here, for simplicity
        x = self.transformer(x)
        logits = self.out(x)
        return logits

# --------- 6. Hyperparams ---------
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 5e-5
NUM_HEADS = 6
NUM_LAYERS = 4
DROPOUT = 0.1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------- 7. Instantiate Model ---------
model = CausalTransformer(
    embedding_matrix=embedding_matrix,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# --------- 8. Dummy DataLoader for testing ---------
# You need to replace this with your real DataFrame loading as per your QA pairs

import pandas as pd
df_dummy = pd.DataFrame({
    'question': ["What is malaria?", "How does gene editing work?"],
    'answer': ["Malaria is a disease.", "Gene editing changes DNA."]
})
train_ds = QADataset(df_dummy, tokenizer, max_len=MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# --------- 9. Training Loop (minimal example) ---------
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs.view(-1, vocab_size), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss / len(train_loader):.4f}")

print("✅ Training complete")


Loading GloVe embeddings...
Tokenizer vocab size: 50258
Embedding matrix ready
Epoch 1/3 - Loss: 10.9593
Epoch 2/3 - Loss: 10.6952
Epoch 3/3 - Loss: 10.4668
✅ Training complete


In [24]:
grid = [
    {"NUM_HEADS": 4, "NUM_LAYERS": 3, "DROPOUT": 0.1, "LR": 3e-5},
    {"NUM_HEADS": 6, "NUM_LAYERS": 4, "DROPOUT": 0.2, "LR": 5e-5},
    # add more if you want
]


In [25]:
best_val_loss = float('inf')
best_config = None
best_model_state = None

for i, config in enumerate(grid):
    print(f"\nTesting config {i+1}: {config}")

    # Initialize model
    model = CausalTransformer(
        embedding_matrix=embedding_matrix,
        num_heads=config["NUM_HEADS"],
        num_layers=config["NUM_LAYERS"],
        max_len=MAX_LEN,
        dropout=config["DROPOUT"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["LR"])
    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    # Train for fewer epochs for tuning speed (e.g. 2)
    model.train()
    for epoch in range(2):
        total_loss = 0
        for input_ids, attention_mask, labels in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs.view(-1, vocab_size), labels.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/2, Train loss: {avg_train_loss:.4f}")

    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:  # You need to prepare val_loader similarly to train_loader
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs.view(-1, vocab_size), labels.view(-1))
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    print(f"Validation loss: {avg_val_loss:.4f}")

    # Save best
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_config = config
        best_model_state = model.state_dict()

print("\nBest config:", best_config)
print(f"Best validation loss: {best_val_loss:.4f}")

# Optionally save best model weights
torch.save(best_model_state, "best_causal_transformer.pth")



Testing config 1: {'NUM_HEADS': 4, 'NUM_LAYERS': 3, 'DROPOUT': 0.1, 'LR': 3e-05}
Epoch 1/2, Train loss: 10.6944


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Epoch 2/2, Train loss: 10.5477


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

...

Epoch 2/2, Train loss: 10.6242


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Validation loss: 10.9447

Best config: {'NUM_HEADS': 6, 'NUM_LAYERS': 4, 'DROPOUT': 0.2, 'LR': 5e-05}
Best validation loss: 10.9447


In [28]:
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)


In [31]:
from evaluate import load

# Load metrics
rouge = load("rouge")
bertscore = load("bertscore")

# Calculate metrics
rouge_result = rouge.compute(predictions=preds, references=refs)
bertscore_result = bertscore.compute(predictions=preds, references=refs, lang="en")

print("ROUGE:", rouge_result)
print("BERTScore:", {
    "precision": sum(bertscore_result['precision']) / len(bertscore_result['precision']),
    "recall": sum(bertscore_result['recall']) / len(bertscore_result['recall']),
    "f1": sum(bertscore_result['f1']) / len(bertscore_result['f1']),
})


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE: {'rouge1': 0.6007057601181882, 'rouge2': 0.59312817876477, 'rougeL': 0.6007532162618621, 'rougeLsum': 0.6013724797907942}
BERTScore: {'precision': 0.8476898857872965, 'recall': 0.9534676372136041, 'f1': 0.8964486761621414}


In [33]:
y_true = [1, 0, 1, 1]
y_prob = [0.95, 0.40, 0.82, 0.89]

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_true, y_prob)
print("AUC:", auc)


AUC: 1.0
